This notebook was made to drill down the ext4 driver of the linux kernel using GDB. It allows precise reproduction of the same commands (same disk state, same PID, etc.).

# Initialization

In [17]:
# Run me!

disk_path="$XDG_RUNTIME_DIR/ext4-disk"
initramfs_path=~/Downloads/linux-debug/linux-minimal-debug/initramfs.cpio.gz
kernel_path=~/Downloads/linux/arch/x86/boot/bzImage
kernel_sources=~/Downloads/linux

function target_exec() {
    ssh root@localhost -p 2222 -o "StrictHostKeyChecking no" $@ 2>/dev/null
}

function target_debugfs() {
    debugfs "$disk_path" -R "$@"
}

# Preparation
1. Create the ext4 disk
2. Start the VM

Necessary kernel parameters:
- CONFIG_DYNAMIC_DEBUG
- CONFIG_GDB_SCRIPTS
- CONFIG_DEBUG_INFO
- CONFIG_DEBUG_INFO_DWARF_TOOLCHAIN_DEFAULT
- CONFIG_DEBUG_INFO_REDUCED=n
- CONFIG_RANDOMIZE_BASE=n
- CONFIG_EXT4_DEBUG
- CONFIG_CHECKPOINT_RESTORE (for pid thing)
- CONFIG_MESSAGE_LOGLEVEL_DEFAULT=7

In [37]:
size=160M

if [ ! -f "$disk_path" ]; then echo "$disk_path is not a regular file"; exit 1; fi

rm "$disk_path"
mkfs.ext4 "$disk_path" $size -O ^has_journal

echo Done

mke2fs 1.47.0 (5-Feb-2023)
Creating regular file /run/user/1000/ext4-disk
Creating filesystem with 40960 4k blocks and 40960 inodes
Filesystem UUID: f7774ec5-1023-4dea-9a69-55fef174fd42
Superblock backups stored on blocks: 
	32768

Allocating group tables: 0/  done                            
Writing inode tables: 0/  done                            
Writing superblocks and filesystem accounting information: 0/  done

Done


In [38]:
# Star the VM
qemu-system-amd64 -kernel "$kernel_path" -append "console=ttyS0 nokaslr" -initrd "$initramfs_path" -hda "$disk_path" -s -device e1000,netdev=net0 -netdev user,id=net0,hostfwd=tcp::2223-:23,hostfwd=tcp::2222-:22 -daemonize -display none -serial file:/tmp/qemu.log

target_exec echo If you see this, the VM is up.

         Automatically detecting the format is dangerous for raw images, write operations on block 0 will be restricted.
         Specify the 'raw' format explicitly to remove the restrictions.
If you see this, the VM is up.


In [6]:
# Shut down cleanly
target_exec poweroff -f

: 255

In [36]:
# Kill the VM
pkill qemu-system-amd

# Debugging

In [19]:
( cd "$kernel_sources" && exec uv tool run gdbfrontend --gdb-args="vmlinux" )&
gdbfrontend_pid=$!
sleep 3 # wait for stdout to finish
echo "127.0.0.1:1234" | xsel -b -i

[1] 2533420
GDBFrontend v0.11.3-beta
Listening on 127.0.0.1: http://127.0.0.1:5550/
Open this address in web browser: http://127.0.0.1:5550/
Opening in existing browser session.


In [86]:
# Stop the debugging server
kill $gdbfrontend_pid

# Filesystem manipulations

In [41]:
target_exec <<EOF
    mkdir mnt
    mount /dev/sda mnt -o defaults,noatime,nodelalloc,nobarrier #,sync
    # turn on debug messages for ext4 module
    echo 'module ext4 +p' > /proc/dynamic_debug/control
EOF

In [85]:
target_exec <<EOF
    force_pid 100
    printf coucou | tee mnt/file_a
    # Theoretically the tee process should be PID 102 since it's at the right of a pipe
    # (try `printf 'echo $$' | sh` to convince yourself)
EOF
target_debugfs "blocks file_a"
echo Should be 33280

#first 32768
#last 40959
# goal 35834

coucou
debugfs 1.47.0 (5-Feb-2023)
33280 
Should be 33280


In [42]:
target_exec <<EOF
    dd bs=64K count=1 if=/dev/urandom of=bigfile
    force_pid 101
    cat bigfile > mnt/file_b
EOF
sleep 1
target_debugfs "blocks file_b"
echo Should be 33280

#first 32768
#last 40959
# goal 35834

debugfs 1.47.0 (5-Feb-2023)
33280 33281 33282 33283 33284 33285 33286 33287 33288 33289 33290 33291 33292 33293 33294 33295 
Should be 33280


In [22]:
target_debugfs "blocks file_b"

debugfs 1.47.0 (5-Feb-2023)

